# QLoRA 4-bit Fine-tune (Unsloth) — Qwen2.5-VL-3B-Instruct

## Mục tiêu
Fine-tune `Qwen2.5-VL-3B-Instruct` thành **English AI/ML/NLP/CV tutor** cho dự án A20.

## Nguồn tham chiếu
- `fine-tune-chatbot/PROPOSAL.md` — chiến lược dataset & lập luận
- `fine-tune-chatbot/PIPELINE.md` — pipeline tổng quan
- `fine-tune-chatbot/plan/02-data-pipeline.md`, `plan/03-finetune.md`, `plan/datasets.md`

## Cấu hình kỹ thuật
| Item | Value |
|---|---|
| Framework | **Unsloth** `FastVisionModel` (≈2× nhanh, 50–70% VRAM so với HF+PEFT) |
| Base | `unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit` (NF4 pre-quantized mirror) |
| Method | QLoRA 4-bit, text-first SFT, **vision tower frozen** trong v1 |
| Train data | Project domain (`question_bank.jsonl` + `units.jsonl`) **+** filtered ELI5 |
| Hardware | ≥ 1 GPU 16GB (T4/L4/A10/4090) |

## Ablation matrix
Theo plan §6, mỗi axis đo một câu hỏi nghiên cứu khác nhau:
1. **Data axis (A/B/C/D)**: Tỷ lệ ELI5 trong mix → tìm điểm cân bằng giữa domain knowledge và explanation style.
2. **Rank axis (8/16/32)**: Capacity của LoRA adapter → tìm rank tối thiểu đủ học domain.
3. **Target modules (attn vs attn+mlp)**: Phạm vi tham số huấn luyện → đo cost-vs-benefit của MLP layers.
4. **Filter axis (raw vs filtered ELI5)**: Chứng minh giá trị của filtering pipeline.

## Selection rule (PIPELINE §9)
Chọn run dựa trên **eval (domain accuracy + explanation quality)**, KHÔNG dựa trên `eval_loss`.
Reject nếu: domain accuracy giảm, hallucination tăng, length tăng nhưng kw_hits giảm.

---
## Section 0 — Setup (Kaggle)

**Mục đích:** Cài Unsloth + dependencies, init seed, auto-detect Kaggle Input dataset.

### Pre-flight checklist (làm trước khi chạy notebook)
1. **Accelerator**: Settings → Accelerator → **GPU T4 x2** hoặc **GPU P100**
2. **Internet**: Settings → Internet → **On** (cần phone-verify account)
3. **Add Input dataset**: Settings → Add Input → chọn dataset chứa `question_bank.jsonl` + `units.jsonl`
4. **Persistence** (optional): Settings → Save Output → Always

### Notes Kaggle-specific
- Working dir: `/kaggle/working/` (persist sau session, ~20GB)
- Input data: read-only mount tại `/kaggle/input/<dataset-slug>/`
- `xformers==0.0.27` để tương thích torch 2.4 preinstalled trên Kaggle
- `CUDA_VISIBLE_DEVICES=0` ép Unsloth dùng 1 GPU (Unsloth không support multi-GPU)
- T4 không support bf16 → notebook tự fallback fp16
- Session timeout: 12h GPU → đủ chạy 4-5 runs ablation

In [ ]:
%%capture
# Kaggle-friendly install. Yêu cầu: Settings > Internet = ON.
# Kaggle có sẵn torch + transformers; chỉ cài thêm Unsloth + deps.
!pip install -q --no-deps bitsandbytes accelerate xformers==0.0.27 peft "trl<0.12.0"
!pip install -q --no-deps unsloth
!pip install -q -U datasets sentencepiece einops qwen-vl-utils
# Eval deps
!pip install -q rouge-score bert-score evaluate

In [ ]:
import os, json, gc, random, re, glob
from pathlib import Path
from dataclasses import dataclass, asdict, field
from typing import Dict, Any, Optional, List
from collections import defaultdict

import torch
import numpy as np
from datasets import load_dataset, Dataset, concatenate_datasets
from unsloth import FastVisionModel, is_bfloat16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# Kaggle: dùng 1 GPU (Unsloth không support multi-GPU)
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

BASE_MODEL = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"

# Kaggle persistent dirs
WORKDIR  = Path("/kaggle/working/qlora_runs"); WORKDIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR = Path("/kaggle/working/eval");        EVAL_DIR.mkdir(parents=True, exist_ok=True)

# ---- Auto-detect canonical artifacts từ Kaggle Input dataset ----
# User upload `question_bank.jsonl` + `units.jsonl` (giữ nguyên hoặc trong thư mục con).
# Tìm tự động trong /kaggle/input/*/.
def _find_first(filename: str) -> Optional[Path]:
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return Path(matches[0]) if matches else None

QB_PATH    = _find_first("question_bank.jsonl")
UNITS_PATH = _find_first("units.jsonl")

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("bf16:", is_bfloat16_supported())
print("QB    :", QB_PATH)
print("Units :", UNITS_PATH)
assert QB_PATH and UNITS_PATH, (
    "Không tìm thấy question_bank.jsonl / units.jsonl trong /kaggle/input/. "
    "Hãy attach Kaggle Dataset chứa 2 file này (Settings > Add Input)."
)

---
## Section 1 — Ablation matrix

**Mục đích:** Khai báo toàn bộ run cấu hình. Mỗi `RunCfg` là 1 ô trong ablation grid.

**4 trục thử nghiệm:**

| Trục | Câu hỏi nghiên cứu | Runs |
|---|---|---|
| `DATA_AXIS` | ELI5 share tối ưu là bao nhiêu? | A=0%, B=10%, C=20%, D=30% |
| `RANK_AXIS` | LoRA rank tối thiểu đủ học domain? | r=8, 16, 32 |
| `TARGET_AXIS` | Có cần fine-tune MLP layers không? | attn-only vs attn+mlp |
| `FILTER_AXIS` | Filtering ELI5 có ích thật không? | raw vs filtered |

**Tham số default (theo plan §3):**
- `lora_r=16, lora_alpha=32` — sweet spot cho 3B model
- `max_seq_len=2048` — đủ cho explanation 450 words
- `lr=2e-4` — chuẩn QLoRA paper
- `epochs=1.0` — tránh overfitting với dataset nhỏ

**Strategy chạy:** Trước tiên chạy `DATA_AXIS` để chốt mix tốt nhất → dùng config đó cho 3 axis còn lại.

In [ ]:
@dataclass
class RunCfg:
    name: str
    eli5_share: float           # 0.0 = domain-only, 1.0 = ELI5-only
    eli5_strict: bool           # áp filter rules
    max_train_samples: int      # cap để chạy nhanh
    lora_r: int = 16
    lora_alpha: int = 32
    target_modules: str = "attn_mlp"
    epochs: float = 1.0
    lr: float = 2e-4
    max_seq_len: int = 2048
    batch_size: int = 2
    grad_accum: int = 8

DATA_AXIS = [
    RunCfg("A_domain_only", 0.00, True, 4000),
    RunCfg("B_eli5_10pct",  0.10, True, 4000),
    RunCfg("C_eli5_20pct",  0.20, True, 4000),
    RunCfg("D_eli5_30pct",  0.30, True, 4000),
]
RANK_AXIS = [
    RunCfg("R_rank8",  0.20, True, 4000, lora_r=8,  lora_alpha=16),
    RunCfg("R_rank16", 0.20, True, 4000, lora_r=16, lora_alpha=32),
    RunCfg("R_rank32", 0.20, True, 4000, lora_r=32, lora_alpha=64),
]
TARGET_AXIS = [
    RunCfg("T_attn_only", 0.20, True, 4000, target_modules="attn"),
    RunCfg("T_attn_mlp",  0.20, True, 4000, target_modules="attn_mlp"),
]
FILTER_AXIS = [
    RunCfg("F_raw",      1.0, False, 4000),
    RunCfg("F_filtered", 1.0, True,  4000),
]

ALL_RUNS = {r.name: r for r in DATA_AXIS + RANK_AXIS + TARGET_AXIS + FILTER_AXIS}
print(f"Total ablation runs: {len(ALL_RUNS)}")

---
## Section 2 — Domain data loaders (project artifacts)

**Mục đích:** Đọc 2 nguồn domain data của dự án và map về schema SFT thống nhất.

### 2.1 Schema gốc

**`question_bank.jsonl`** — 1276 MCQ items:
```json
{"course_id":"CS224n", "lecture_id":"lecture-01", "question":"...",
 "choices":["A","B","C","D"], "answer_index":0, "explanation":"...",
 "difficulty":"easy", "question_intent":"conceptual", ...}
```

**`units.jsonl`** — 376 lecture units:
```json
{"course_id":"CS224n", "lecture_id":"lecture-10", "unit_name":"Why ... attention",
 "summary":"...", "key_points":[{"text":"...","timestamp_s":278},...]}
```

### 2.2 Hai cách convert MCQ → SFT

Mỗi MCQ tạo **2 training examples** để mô hình học cả 2 năng lực:

1. **MCQ-format** (test-the-model): User đưa câu hỏi + 4 choices, assistant trả lời `"The answer is (B). {explanation}"`. Dạy mô hình answer trắc nghiệm — match format eval.
2. **Open-form** (teach-the-concept): User chỉ đưa câu hỏi, assistant trả lời bằng `explanation`. Dạy mô hình giải thích khái niệm dạng tutor.

### 2.3 Units → SFT

Mỗi unit tạo 1 example: User hỏi `"Explain {lecture_title}: {unit_name}"`, assistant trả về `summary + bullet key_points`.

### 2.4 Metadata giữ lại

`lecture_id` cực kỳ quan trọng — Section 4 dùng để split train/val/test theo lecture (tránh leakage giữa các MCQ cùng 1 bài giảng).

In [ ]:
LETTERS = ["A","B","C","D","E","F"]

def mcq_to_chat(item: Dict[str,Any]) -> List[Dict[str,Any]]:
    """Convert 1 MCQ → 2 chat samples (mcq-format + open-form)."""
    q = item["question"].strip()
    choices = item["choices"]
    idx = item["answer_index"]
    expl = (item.get("explanation") or "").strip()
    if idx is None or idx >= len(choices):
        return []
    correct_letter = LETTERS[idx]

    # 1) MCQ-format
    choice_text = "\n".join(f"({LETTERS[i]}) {c}" for i,c in enumerate(choices))
    user1 = f"{q}\n\n{choice_text}\n\nWhich option is correct?"
    asst1 = f"The correct answer is ({correct_letter}). {expl}".strip()

    # 2) Open-form
    user2 = q
    asst2 = expl if expl else choices[idx]

    base = {"lecture_id": item.get("lecture_id","unknown"), "course_id": item.get("course_id",""), "src": "qb"}
    return [
        {"user": user1, "assistant": asst1, **base, "variant":"mcq"},
        {"user": user2, "assistant": asst2, **base, "variant":"open"},
    ]

def unit_to_chat(item: Dict[str,Any]) -> Optional[Dict[str,Any]]:
    """Convert 1 unit → 1 chat sample (concept explanation)."""
    title = item.get("lecture_title") or item.get("lecture_id","")
    name = item.get("unit_name") or ""
    summary = (item.get("summary") or "").strip()
    if not summary:
        return None
    kps = item.get("key_points") or []
    bullets = "\n".join(f"- {kp['text']}" for kp in kps if kp.get("text"))
    user = f"Explain this lecture topic: {title} — {name}"
    asst = summary + (("\n\nKey points:\n" + bullets) if bullets else "")
    return {"user": user, "assistant": asst,
            "lecture_id": item.get("lecture_id","unknown"),
            "course_id": item.get("course_id",""),
            "src": "units", "variant": "summary"}

def load_domain_dataset() -> Dataset:
    rows = []
    if QB_PATH.exists():
        for line in open(QB_PATH, encoding="utf-8"):
            try: obj = json.loads(line)
            except json.JSONDecodeError: continue
            if not obj.get("qa_gate_passed", True): continue   # bỏ items chưa pass QA gate
            rows.extend(mcq_to_chat(obj))
    if UNITS_PATH.exists():
        for line in open(UNITS_PATH, encoding="utf-8"):
            try: obj = json.loads(line)
            except json.JSONDecodeError: continue
            if not obj.get("active", True): continue
            r = unit_to_chat(obj)
            if r: rows.append(r)
    return Dataset.from_list(rows)

DOMAIN_DS = load_domain_dataset()
print("Domain SFT samples:", len(DOMAIN_DS))
print("By source:", {s: sum(1 for r in DOMAIN_DS if r['src']==s) for s in ['qb','units']})
print("By variant:", {v: sum(1 for r in DOMAIN_DS if r['variant']==v) for v in ['mcq','open','summary']})

---
## Section 3 — ELI5 filtering pipeline

**Mục đích:** Lọc ELI5 thành **explanation-style auxiliary data**. KHÔNG dùng raw ELI5 (PROPOSAL §7).

### 3.1 Filter rules (plan/datasets.md §Tier 2)

| Group | Rule | Lý do |
|---|---|---|
| Question type | Bắt đầu bằng `why / how / what happens / how does / difference` | Đảm bảo style "giải thích" |
| Answer length | 120–450 từ | Tránh quá ngắn (1-liner) hoặc quá dài (rambling) |
| Topic whitelist | ML / DL / math / CS / probability / optimization / science | Bám AI/ML/NLP/CV scope |
| Exclude | politics / sports / celebrity / entertainment | Loại noise generic |

### 3.2 Tại sao split rule này quan trọng

PROPOSAL §7 nói: "raw ELI5 too generic → causes domain drift". Filter này không nhằm tăng quantity, mà nhằm **chỉ giữ phong cách giải thích phù hợp**.

### 3.3 Output schema

Chuẩn hoá về cùng schema với domain data: `{user, assistant, lecture_id='eli5', course_id='', src='eli5', variant='style'}`.

In [ ]:
QTYPE_RX = re.compile(r"^\s*(why|how|what happens|how does|how do|what is the difference|difference between)\b", re.I)
TOPIC_KW = ["machine learning","deep learning","neural network","transformer","embedding",
    "language model","computer vision","cnn","convolution","segmentation","detection",
    "probability","statistic","linear algebra","optimization","gradient","backprop",
    "algorithm","computation","information theory","science","physics","chemistry",
    "biology","math","computer","software","data"]
EXCLUDE_KW = ["trump","biden","election","politic","celebrity","kardashian",
    "basketball","nba","nfl","soccer match","movie","tv show","hollywood"]

def _has_any(t, kws):
    tl = t.lower(); return any(k in tl for k in kws)

def _eli5_keep(q: str, a: str, strict: bool) -> bool:
    if not q or not a: return False
    if not strict: return True
    if not QTYPE_RX.search(q): return False
    wc = len(a.split())
    if wc < 120 or wc > 450: return False
    if _has_any(q + " " + a, EXCLUDE_KW): return False
    if not _has_any(q + " " + a, TOPIC_KW): return False
    return True

def load_filtered_eli5(strict: bool, max_n: int) -> Dataset:
    raw = load_dataset("sentence-transformers/eli5", split="train")
    rows = []
    for ex in raw:
        q = ex.get("question") or ex.get("title") or ""
        a = ex.get("answer") or ex.get("response") or ""
        if _eli5_keep(q, a, strict):
            rows.append({"user": q, "assistant": a,
                         "lecture_id": "eli5", "course_id": "",
                         "src": "eli5", "variant": "style"})
        if len(rows) >= max_n * 2:  # buffer trước khi shuffle
            break
    ds = Dataset.from_list(rows)
    if len(ds) > max_n:
        ds = ds.shuffle(seed=SEED).select(range(max_n))
    return ds

_smoke = load_filtered_eli5(strict=True, max_n=200)
print("filtered ELI5 sample:", len(_smoke), "\nex:", _smoke[0]["user"][:120])

---
## Section 4 — Train/val/test split & mix builder

**Mục đích:** Chia domain data theo `lecture_id` (PIPELINE §5) và build mix theo cấu hình ablation.

### 4.1 Split policy — split by `lecture_id`, KHÔNG split random

Lý do: 1 lecture có nhiều MCQ + units; nếu split random, train có thể chứa MCQ q1 và test chứa MCQ q2 cùng lecture → **leakage**. Split by lecture đảm bảo test set là lectures hoàn toàn unseen.

**Tỷ lệ:** Train 80% / Val 10% / Test 10% (theo PIPELINE §5).

### 4.2 Build mix per ablation run

Cho `RunCfg(eli5_share=s, max_train_samples=N)`:
- ELI5 chiếm `s × N` samples
- Domain chiếm `(1-s) × N` samples (lấy từ TRAIN split, không động vào VAL/TEST)
- Nếu domain không đủ, bù bằng filtered ELI5 (đánh dấu trong manifest)

### 4.3 Outputs
- `DOMAIN_TRAIN`, `DOMAIN_VAL`, `DOMAIN_TEST` — split theo lecture, dùng chung cho mọi run
- `build_mix(cfg)` → `(train_ds, val_ds_for_loss)` cho training
- `DOMAIN_TEST` được giữ riêng cho **benchmark Section 9**

In [ ]:
def split_by_lecture(ds: Dataset, val_ratio=0.10, test_ratio=0.10):
    """Split theo lecture_id để tránh leakage giữa train/val/test."""
    lectures = sorted({r["lecture_id"] for r in ds if r["lecture_id"] != "eli5"})
    rng = random.Random(SEED); rng.shuffle(lectures)
    n = len(lectures)
    n_test = max(1, int(n * test_ratio))
    n_val  = max(1, int(n * val_ratio))
    test_l = set(lectures[:n_test])
    val_l  = set(lectures[n_test:n_test+n_val])
    train_l = set(lectures[n_test+n_val:])
    def _bucket(r):
        lid = r["lecture_id"]
        if lid in test_l:  return "test"
        if lid in val_l:   return "val"
        return "train"
    by = defaultdict(list)
    for r in ds: by[_bucket(r)].append(r)
    return (Dataset.from_list(by["train"]),
            Dataset.from_list(by["val"]),
            Dataset.from_list(by["test"]),
            {"train_lectures": sorted(train_l), "val_lectures": sorted(val_l), "test_lectures": sorted(test_l)})

DOMAIN_TRAIN, DOMAIN_VAL, DOMAIN_TEST, SPLIT_INFO = split_by_lecture(DOMAIN_DS)
(EVAL_DIR/"split_info.json").write_text(json.dumps(SPLIT_INFO, indent=2))
print("Train/Val/Test:", len(DOMAIN_TRAIN), len(DOMAIN_VAL), len(DOMAIN_TEST))
print("Test lectures (unseen):", SPLIT_INFO["test_lectures"][:5], "...")

In [ ]:
SYSTEM = ("You are an AI/ML/NLP/CV tutor. Explain concepts clearly, accurately, "
          "and step by step in English.")

def to_chat(ex):
    return {"messages": [
        {"role":"system",   "content":[{"type":"text","text":SYSTEM}]},
        {"role":"user",     "content":[{"type":"text","text":ex["user"]}]},
        {"role":"assistant","content":[{"type":"text","text":ex["assistant"]}]},
    ]}

def build_mix(cfg: RunCfg):
    N = cfg.max_train_samples
    n_eli5 = int(round(N * cfg.eli5_share))
    n_dom = N - n_eli5
    parts = []

    if n_dom > 0 and len(DOMAIN_TRAIN) > 0:
        dom = DOMAIN_TRAIN.shuffle(seed=SEED).select(range(min(n_dom, len(DOMAIN_TRAIN))))
        parts.append(dom)
        deficit = n_dom - len(dom)
    else:
        deficit = n_dom

    if n_eli5 + deficit > 0:
        parts.append(load_filtered_eli5(cfg.eli5_strict, n_eli5 + deficit))

    full = concatenate_datasets(parts).shuffle(seed=SEED)
    train = full.map(to_chat, remove_columns=full.column_names)
    val   = DOMAIN_VAL.map(to_chat, remove_columns=DOMAIN_VAL.column_names) if len(DOMAIN_VAL) else None
    return train, val

_tr,_va = build_mix(DATA_AXIS[2])
print("mix C: train=", len(_tr), " val=", 0 if _va is None else len(_va))

---
## Section 5 — Load model + attach LoRA (Unsloth)

**Mục đích:** Load Qwen2.5-VL-3B 4-bit và gắn LoRA adapter theo cấu hình.

### 5.1 Vì sao Unsloth
- `FastVisionModel.from_pretrained` đã tích hợp NF4 + double-quant + bf16 compute trong 1 dòng.
- `use_gradient_checkpointing="unsloth"` (chứ không phải `True`) — Unsloth có version tối ưu hơn HF default, tiết kiệm thêm ~30% VRAM.
- Patch xLA RoPE và RMSNorm bằng Triton kernels → ~2× throughput.

### 5.2 Vision tower frozen
PIPELINE §3 quy định v1 freeze vision encoder. `finetune_vision_layers=False` đảm bảo vision path còn nguyên cho deployment multimodal sau này.

### 5.3 Target modules
- `attn`: chỉ Q/K/V/O — học attention pattern, ít tham số (~0.3% model)
- `attn_mlp`: thêm gate/up/down — học cả MLP, nhiều tham số hơn (~0.7% model), thường tốt hơn cho domain knowledge

In [ ]:
TARGET_MODULES = {
    "attn":     ["q_proj","k_proj","v_proj","o_proj"],
    "attn_mlp": ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
}

def load_model(cfg: RunCfg):
    model, tok = FastVisionModel.from_pretrained(
        BASE_MODEL,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
        max_seq_length=cfg.max_seq_len,
    )
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=(cfg.target_modules == "attn_mlp"),
        r=cfg.lora_r, lora_alpha=cfg.lora_alpha, lora_dropout=0.05,
        bias="none", random_state=SEED,
        target_modules=TARGET_MODULES[cfg.target_modules],
    )
    return model, tok

---
## Section 6 — Train one ablation run

**Mục đích:** Wrapper chạy 1 cell trong ablation grid: build mix → load model → SFT → save adapter + manifest.

### 6.1 Hyperparams quan trọng
- `optim="adamw_8bit"` — quantized optimizer của bitsandbytes, tiết kiệm ~50% VRAM cho optimizer states
- `lr_scheduler="cosine" + warmup 3%` — chuẩn QLoRA paper (Dettmers 2023)
- `bf16` nếu Ampere+, `fp16` fallback cho T4
- `eval_strategy="steps", eval_steps=100` — track loss curve, giúp detect overfitting sớm

### 6.2 Unsloth-specific args
- `remove_unused_columns=False` — Unsloth collator cần raw `messages` field
- `dataset_text_field=""` + `skip_prepare_dataset=True` — bỏ qua TRL preprocessing, để Unsloth tự collate

### 6.3 Manifest
Lưu cấu hình, sample counts, domain usage để reproducible. Đặt cùng folder với adapter để debug sau.

In [ ]:
def train_run(cfg: RunCfg):
    out = WORKDIR / cfg.name; out.mkdir(parents=True, exist_ok=True)
    print(f"\n========== RUN: {cfg.name} ==========")
    print(json.dumps(asdict(cfg), indent=2))

    train_ds, val_ds = build_mix(cfg)
    (out/"manifest.json").write_text(json.dumps({
        "cfg": asdict(cfg), "n_train": len(train_ds),
        "n_val": 0 if val_ds is None else len(val_ds),
        "domain_size": len(DOMAIN_TRAIN),
    }, indent=2))

    model, tok = load_model(cfg)
    FastVisionModel.for_training(model)

    args = SFTConfig(
        output_dir=str(out),
        num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.batch_size,
        per_device_eval_batch_size=cfg.batch_size,
        gradient_accumulation_steps=cfg.grad_accum,
        learning_rate=cfg.lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps" if val_ds is not None else "no",
        eval_steps=100,
        save_strategy="epoch", save_total_limit=1,
        report_to="none",
        max_seq_length=cfg.max_seq_len,
        optim="adamw_8bit", seed=SEED,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    )
    trainer = SFTTrainer(
        model=model, tokenizer=tok,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=UnslothVisionDataCollator(model, tok),
        args=args,
    )
    trainer.train()
    model.save_pretrained(str(out/"adapter"))
    tok.save_pretrained(str(out/"adapter"))
    metrics = trainer.evaluate() if val_ds is not None else {}
    (out/"eval_loss.json").write_text(json.dumps(metrics, indent=2, default=str))
    del trainer, model; gc.collect(); torch.cuda.empty_cache()
    return metrics

---
## Section 7 — Run the ablation matrix

**Mục đích:** Loop qua các runs đã chọn. Default chạy `DATA_AXIS` (4 runs) — đủ để chốt mix tốt nhất.

**Strategy đề xuất:**
1. Chạy `DATA_AXIS` trước → chấm Section 9 → chốt best mix (vd `C_eli5_20pct`).
2. Update `RANK_AXIS` / `TARGET_AXIS` / `FILTER_AXIS` để dùng best mix làm baseline.
3. Chạy 3 axis còn lại tuần tự.

**Total time estimate:** ~30-45 phút/run trên 4090 với 4000 samples. 11 runs ≈ 6-8h.

In [ ]:
RUN_THESE = ["A_domain_only","B_eli5_10pct","C_eli5_20pct","D_eli5_30pct"]
# RUN_THESE += ["R_rank8","R_rank16","R_rank32"]
# RUN_THESE += ["T_attn_only","T_attn_mlp"]
# RUN_THESE += ["F_raw","F_filtered"]

results = {}
for name in RUN_THESE:
    try:
        results[name] = train_run(ALL_RUNS[name])
    except Exception as e:
        print(f"[FAIL] {name}: {e}")
        results[name] = {"error": str(e)}
(WORKDIR/"all_results.json").write_text(json.dumps(results, indent=2, default=str))
results

---
## Section 8 — Inference helper

**Mục đích:** Load adapter + base model để generate. Dùng cho cả benchmark và spot-check.

### 8.1 Inference mode
`FastVisionModel.for_inference(model)` bật:
- KV cache (use_cache=True)
- Disable gradient checkpointing
- Patch generate cho speed

### 8.2 Decoding
Mặc định `do_sample=False, temperature=0.0` (greedy) — deterministic, cần thiết cho benchmark.
Spot-check thì có thể `temperature=0.7, top_p=0.9` cho diversity.

In [ ]:
def load_with_adapter(run_name: str, max_seq_len=2048):
    model, tok = FastVisionModel.from_pretrained(
        str(WORKDIR/run_name/"adapter"),
        load_in_4bit=True, max_seq_length=max_seq_len,
    )
    FastVisionModel.for_inference(model)
    return model, tok

@torch.no_grad()
def generate(model, tok, user_q: str, max_new_tokens=400, greedy=True):
    msgs = [
        {"role":"system","content":[{"type":"text","text":SYSTEM}]},
        {"role":"user",  "content":[{"type":"text","text":user_q}]},
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inp, max_new_tokens=max_new_tokens,
        do_sample=not greedy,
        temperature=0.0 if greedy else 0.7,
        top_p=1.0 if greedy else 0.9,
        repetition_penalty=1.05,
    )
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

---
## Section 9 — Benchmark suite

Theo `PROPOSAL.md` §12 và `PIPELINE.md` §8, benchmark chia 5 tier:

| Tier | Bench | Vai trò | Metric |
|---|---|---|---|
| **A** | Internal MCQ held-out (`DOMAIN_TEST`) | **Shipping gate** — quan trọng nhất | Accuracy |
| **B1** | MMLU subset (ML/CS/Math/Stats) | Academic regression | Accuracy |
| **B2** | MMLU-Pro | Reasoning regression | Accuracy |
| **B3** | TheoremQA | Technical reasoning | Accuracy |
| **C** | ELI5 dev/test (filtered) | Style only | ROUGE-L + BERTScore |

### 9.1 Selection rule (hard-coded từ plan §9)
```
ACCEPT iff:
  internal_acc(run) > internal_acc(base) + 3pt
  AND mmlu_drop ≤ 2pt (so với base)
  AND bertscore_eli5 ≥ baseline
REJECT if:
  internal_acc giảm
  OR length tăng nhưng kw_hits giảm
```

### 9.2 Vì sao 5 tier
- Tier A: bám đúng bài toán → quyết định ship.
- Tier B: regression check → đảm bảo fine-tune không phá kiến thức tổng quát.
- Tier C: style → đo cải thiện về explanation quality (fluency, coherence).

### 9.A — Tier A: Internal MCQ accuracy

**Method:** Cho mỗi MCQ trong `DOMAIN_TEST` (variant=`mcq`), prompt model với 4 choices A/B/C/D, parse letter từ output, so với `answer_index`.

**Metric:** Accuracy = correct / total. Break down theo `course_id` để xem có course nào tệ.

**Tại sao đây là gate:** Match đúng bài toán deployment (chatbot trả lời MCQ trong app). Cao hơn MMLU vì sát domain hơn.

In [ ]:
ANS_RX = re.compile(r"\b(?:answer\s+is|correct\s+answer\s+is|answer:)\s*\(?([A-F])\)?", re.I)
ANS_FALLBACK = re.compile(r"^\s*\(?([A-F])\)", re.I)

def parse_letter(text: str) -> Optional[str]:
    m = ANS_RX.search(text) or ANS_FALLBACK.search(text.strip())
    return m.group(1).upper() if m else None

def benchmark_internal_mcq(run_name: str, max_items=200):
    """Tier A — internal held-out MCQ accuracy."""
    mcq_test = [r for r in DOMAIN_TEST if r["variant"] == "mcq"][:max_items]
    if not mcq_test:
        print("No MCQ items in test split."); return None
    model, tok = load_with_adapter(run_name)
    correct = 0; per_course = defaultdict(lambda: [0,0])
    rows = []
    for r in mcq_test:
        out = generate(model, tok, r["user"], max_new_tokens=200)
        pred = parse_letter(out)
        gold = r["assistant"].split("(",1)[1][0] if "(" in r["assistant"] else None
        ok = (pred == gold)
        correct += int(ok)
        per_course[r["course_id"]][0] += int(ok); per_course[r["course_id"]][1] += 1
        rows.append({"q": r["user"][:200], "pred": pred, "gold": gold, "ok": ok, "course": r["course_id"]})
    acc = correct / len(mcq_test)
    by_course = {c: round(v[0]/v[1], 3) for c,v in per_course.items()}
    out = {"run": run_name, "tier": "A_internal_mcq", "n": len(mcq_test),
           "accuracy": round(acc,4), "by_course": by_course}
    (EVAL_DIR/f"{run_name}_tierA.json").write_text(json.dumps(out, indent=2))
    with open(EVAL_DIR/f"{run_name}_tierA_details.jsonl","w",encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r,ensure_ascii=False)+"\n")
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

### 9.B — Tier B: MMLU + MMLU-Pro + TheoremQA

**Method:** Loglikelihood scoring — cho mỗi choice, tính `log P(choice | prompt)`, chọn argmax. Chuẩn của `lm-eval-harness`.

Trong notebook này dùng simple greedy generation + parse letter (đơn giản hơn, kết quả gần tương đương cho instruction-tuned models). Nếu cần publication-grade, switch sang `lm-eval-harness`.

**MMLU subjects** (theo PROPOSAL §12.3):
- `machine_learning` — bám sát nhất
- `college_computer_science`
- `college_mathematics`
- `high_school_statistics`

**Pass criteria:** Drop ≤ 2pt so với base (chưa fine-tune).

In [ ]:
MMLU_SUBJECTS = ["machine_learning","college_computer_science",
                 "college_mathematics","high_school_statistics"]

def _format_mmlu(ex):
    choices = "\n".join(f"({LETTERS[i]}) {c}" for i,c in enumerate(ex["choices"]))
    return f"{ex['question']}\n\n{choices}\n\nWhich option is correct?"

def benchmark_mmlu(run_name: str, subjects=MMLU_SUBJECTS, max_per_subject=50):
    model, tok = load_with_adapter(run_name)
    results = {}
    for subj in subjects:
        try:
            ds = load_dataset("cais/mmlu", subj, split="test")
        except Exception as e:
            print(f"[skip {subj}]: {e}"); continue
        if len(ds) > max_per_subject:
            ds = ds.shuffle(seed=SEED).select(range(max_per_subject))
        ok = 0
        for ex in ds:
            pred = parse_letter(generate(model, tok, _format_mmlu(ex), max_new_tokens=50))
            ok += int(pred == LETTERS[ex["answer"]])
        results[subj] = {"n": len(ds), "acc": round(ok/len(ds), 4)}
    out = {"run": run_name, "tier": "B1_mmlu", "subjects": results,
           "avg_acc": round(np.mean([r["acc"] for r in results.values()]), 4) if results else 0.0}
    (EVAL_DIR/f"{run_name}_tierB1.json").write_text(json.dumps(out, indent=2))
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

def benchmark_mmlu_pro(run_name: str, max_items=100):
    model, tok = load_with_adapter(run_name)
    try:
        ds = load_dataset("TIGER-Lab/MMLU-Pro", split="test")
    except Exception as e:
        print(f"[mmlu-pro skip]: {e}"); del model; return None
    # filter category liên quan
    keep = {"computer science","math","engineering","physics"}
    ds = ds.filter(lambda x: x.get("category","").lower() in keep)
    if len(ds) > max_items: ds = ds.shuffle(seed=SEED).select(range(max_items))
    ok = 0
    for ex in ds:
        opts = ex["options"]
        choices = "\n".join(f"({LETTERS[i]}) {c}" for i,c in enumerate(opts))
        prompt = f"{ex['question']}\n\n{choices}\n\nWhich option is correct?"
        pred = parse_letter(generate(model, tok, prompt, max_new_tokens=80))
        ok += int(pred == ex["answer"])
    out = {"run": run_name, "tier": "B2_mmlu_pro", "n": len(ds),
           "accuracy": round(ok/len(ds),4)}
    (EVAL_DIR/f"{run_name}_tierB2.json").write_text(json.dumps(out, indent=2))
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

def benchmark_theoremqa(run_name: str, max_items=100):
    model, tok = load_with_adapter(run_name)
    try:
        ds = load_dataset("TIGER-Lab/TheoremQA", split="test")
    except Exception as e:
        print(f"[theoremqa skip]: {e}"); del model; return None
    if len(ds) > max_items: ds = ds.shuffle(seed=SEED).select(range(max_items))
    # TheoremQA dạng free-form numeric → exact match đơn giản
    correct = 0
    for ex in ds:
        pred = generate(model, tok, ex["Question"], max_new_tokens=200).strip()
        gold = str(ex.get("Answer","")).strip().lower()
        if gold and gold in pred.lower(): correct += 1
    out = {"run": run_name, "tier": "B3_theoremqa", "n": len(ds),
           "loose_match_acc": round(correct/len(ds),4)}
    (EVAL_DIR/f"{run_name}_tierB3.json").write_text(json.dumps(out, indent=2))
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

### 9.C — Tier C: ELI5 explanation style (ROUGE-L + BERTScore)

**Method:** Generate explanation cho 50 ELI5 dev questions → so với reference answers.

**Metrics:**
- **ROUGE-L F1** — overlap ngữ pháp với reference. Thấp nhưng nên ≥ baseline.
- **BERTScore F1** (`microsoft/deberta-xlarge-mnli`) — semantic similarity. Robust hơn ROUGE.
- **Avg length** — kiểm tra mô hình không dài lê thê hoặc cụt lủn.

**Lưu ý:** ELI5 reference không phải ground truth tuyệt đối (Reddit data có nhiễu). Metric này chỉ đo "phong cách" — không reject run dựa trên ROUGE thấp.

In [ ]:
def benchmark_eli5_style(run_name: str, max_items=50):
    from rouge_score import rouge_scorer
    from bert_score import score as bert_score

    eli5_dev = load_filtered_eli5(strict=True, max_n=max_items)
    model, tok = load_with_adapter(run_name)
    preds, refs, lens = [], [], []
    for r in eli5_dev:
        out = generate(model, tok, r["user"], max_new_tokens=400)
        preds.append(out); refs.append(r["assistant"]); lens.append(len(out.split()))
    del model; gc.collect(); torch.cuda.empty_cache()

    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    rouge_l = np.mean([rouge.score(r,p)["rougeL"].fmeasure for p,r in zip(preds,refs)])
    P,R,F = bert_score(preds, refs, lang="en", verbose=False)
    out = {"run": run_name, "tier": "C_eli5_style", "n": len(preds),
           "rouge_l_f1": round(float(rouge_l),4),
           "bertscore_f1": round(float(F.mean()),4),
           "avg_words": round(float(np.mean(lens)),1)}
    (EVAL_DIR/f"{run_name}_tierC.json").write_text(json.dumps(out, indent=2))
    return out

### 9.D — Run full benchmark suite per run

**Mục đích:** 1 lệnh chạy cả 5 tier cho 1 run.

**Time estimate:** ~15-20 phút/run cho 200 internal MCQ + 4×50 MMLU + 100 MMLU-Pro + 100 TheoremQA + 50 ELI5.

In [ ]:
def run_full_benchmark(run_name: str):
    print(f"\n##### BENCHMARK {run_name} #####")
    res = {
        "A":  benchmark_internal_mcq(run_name),
        "B1": benchmark_mmlu(run_name),
        "B2": benchmark_mmlu_pro(run_name),
        "B3": benchmark_theoremqa(run_name),
        "C":  benchmark_eli5_style(run_name),
    }
    (EVAL_DIR/f"{run_name}_full.json").write_text(json.dumps(res, indent=2))
    return res

# for n in RUN_THESE: run_full_benchmark(n)

---
## Section 10 — Compare runs & apply selection rule

**Mục đích:** Tổng hợp metrics, áp selection rule, chọn best run.

### 10.1 Bảng tổng hợp
Một dòng = 1 run. Cột:
- `tierA_acc` — internal MCQ (gate chính)
- `tierB1_acc` — MMLU avg
- `tierB2_acc` — MMLU-Pro
- `tierC_bertscore` — ELI5 semantic similarity
- `tierC_avg_words` — length sanity

### 10.2 Selection rule
Implement nguyên văn từ PIPELINE §9:
```
ACCEPT iff:  tierA > base+3pt  AND  mmlu_drop ≤ 2pt  AND  bertscore ≥ baseline
REJECT if:   tierA ↓  OR  length ↑ kèm bertscore ↓
```
Cần baseline = chạy `run_full_benchmark` trên base model (`unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit` chưa fine-tune).

In [ ]:
import pandas as pd

def summarize(runs=None):
    runs = runs or RUN_THESE
    rows = []
    for name in runs:
        r = {"run": name}
        for tier, key, dst in [
            ("tierA", "accuracy", "tierA_acc"),
            ("tierB1","avg_acc",  "tierB1_acc"),
            ("tierB2","accuracy", "tierB2_acc"),
            ("tierB3","loose_match_acc","tierB3_acc"),
            ("tierC", "bertscore_f1","tierC_bertscore"),
            ("tierC", "rouge_l_f1", "tierC_rouge"),
            ("tierC", "avg_words",  "tierC_avg_words"),
        ]:
            p = EVAL_DIR / f"{name}_{tier}.json"
            if p.exists(): r[dst] = json.loads(p.read_text()).get(key)
        lp = WORKDIR/name/"eval_loss.json"
        if lp.exists(): r["eval_loss"] = json.loads(lp.read_text()).get("eval_loss")
        rows.append(r)
    df = pd.DataFrame(rows)
    df.to_csv(EVAL_DIR/"run_summary.csv", index=False)
    return df

def apply_selection_rule(df: pd.DataFrame, baseline_row: Dict[str,float]):
    """baseline_row: dict với tierA_acc, tierB1_acc, tierC_bertscore của base model."""
    verdicts = []
    for _, r in df.iterrows():
        reasons = []
        if r.get("tierA_acc",0) < baseline_row["tierA_acc"] + 0.03:
            reasons.append("tierA improvement < 3pt")
        if r.get("tierB1_acc",0) < baseline_row["tierB1_acc"] - 0.02:
            reasons.append("MMLU drop > 2pt")
        if r.get("tierC_bertscore",0) < baseline_row["tierC_bertscore"]:
            reasons.append("BERTScore regression")
        verdicts.append("ACCEPT" if not reasons else f"REJECT: {'; '.join(reasons)}")
    df = df.copy(); df["verdict"] = verdicts
    return df

# df = summarize()
# df = apply_selection_rule(df, baseline_row={"tierA_acc":0.45,"tierB1_acc":0.55,"tierC_bertscore":0.82})
# df

---
## Section 11 — Export options

**Mục đích:** Sau khi chọn best run, export adapter cho deployment.

### 11.1 Merge LoRA → 16bit weights
Khi cần deploy không cần PEFT runtime (vd vLLM, SGLang).

### 11.2 GGUF cho llama.cpp
**Lưu ý:** GGUF chỉ export được text path. Vision encoder không support GGUF → mất khả năng vision.
Nếu cần vision tại deploy, dùng vLLM với merged 16bit thay vì GGUF.

### 11.3 Push lên HuggingFace Hub
`model.push_to_hub_merged("username/qwen25vl-tutor", tok, save_method="merged_16bit")`.

In [ ]:
# BEST_RUN = "C_eli5_20pct"  # đổi sau khi xem summary
# model, tok = load_with_adapter(BEST_RUN)
#
# # Option A — merged 16bit (cho vLLM/SGLang)
# model.save_pretrained_merged("qwen25vl-tutor-merged", tok, save_method="merged_16bit")
#
# # Option B — GGUF text-only (llama.cpp)
# # model.save_pretrained_gguf("qwen25vl-tutor-gguf", tok, quantization_method="q4_k_m")
#
# # Option C — push to HF Hub
# # model.push_to_hub_merged("YOUR_USER/qwen25vl-tutor", tok, save_method="merged_16bit", token="hf_...")

---
## Section 12 — Notes & next steps

### Đã làm
- ✅ QLoRA 4-bit Unsloth pipeline (Kaggle-ready)
- ✅ Domain data: MCQ + units → 3 SFT variants
- ✅ Filtered ELI5 auxiliary corpus
- ✅ Split by `lecture_id` (no leakage)
- ✅ Ablation matrix 4 trục
- ✅ Benchmark 5-tier (Internal MCQ, MMLU, MMLU-Pro, TheoremQA, ELI5 style)
- ✅ Selection rule auto-apply

### v2 ideas (để dành sau)
- **Vision unfreeze:** đổi `finetune_vision_layers=True` + thêm dataset multimodal (slide screenshots với MCQ).
- **LLM-judge eval:** dùng Claude/GPT-4o chấm 5 axes (correctness, completeness, clarity, grounding, coherence) thay heuristic.
- **DPO/KTO:** sau SFT, RLHF với preference data từ `qa_history.jsonl` (user feedback).
- **lm-eval-harness integration:** chuyển benchmarks sang harness chuẩn.
- **Multi-turn:** thêm follow-up Q&A từ `qa_history.jsonl`.

### Trouble-shooting (Kaggle)
| Issue | Fix |
|---|---|
| `assert QB_PATH and UNITS_PATH` fail | Settings → Add Input → attach Kaggle Dataset của bạn |
| `pip install unsloth` chậm/timeout | Bật Internet ở Settings; rerun cell 0 |
| `xformers` version conflict | Restart kernel sau khi pip install (Run All lại từ đầu) |
| OOM trên T4 (16GB) | Giảm `max_seq_len=1024`, `batch_size=1`, hoặc `lora_r=8` |
| `bf16` error | T4 không support → notebook đã auto fallback fp16, nếu vẫn lỗi check `is_bfloat16_supported()` |
| `trl` API mismatch | Kernel cache cũ; restart và rerun cell 0 |
| ELI5 dataset 404 | Fallback: `KennethEnevoldsen/eli5-category` |
| MMLU-Pro empty | Mở rộng category whitelist trong `benchmark_mmlu_pro` |
| Session timeout 12h | Save adapter sau mỗi run; chạy benchmark trong session mới |
| Output không save | Settings → Save Output → Always; hoặc download `/kaggle/working/qlora_runs/` cuối session |